# Compute Diversity Metrics for POIs

This notebook computes diversity metrics for each POI across three contexts:

1. **Residential**: Demographic composition of the tract/DeSO where POI is located
2. **Transit Catchment**: Composition of population reachable within 45 min by transit
3. **Actual Visitors**: Composition based on foot traffic visitor home locations

**Two metric sets:**
- **Set 1 (Entropy)**: Multi-category evenness (4 income quartiles, binary birth)
- **Set 2 (ICE)**: Bipolar concentration (Q4 vs Q1, native vs foreign)

**Countries:**
- Sweden: 15 southern counties (with city filtering)
- US: Cities with transit catchment data

In [2]:
%cd /workspace

import sys
sys.path.insert(0, 'src')

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import json
from tqdm import tqdm

from features.diversity_metrics import (
    entropy_income_quartiles,
    entropy_birth_binary,
    ice_income,
    ice_birth,
    compute_diversity_metrics,
)

# Directories
DESO_DIR = Path('dbs/deso')
US_CENSUS_DIR = Path('dbs/us_census')
US_CENSUS_GEO_DIR = Path('dbs/census_geo')
ROUTING_DIR = Path('dbs/routing')
SE_TRAFFIC_DIR = Path('dbs/sweden_weekly_patterns')
US_TRAFFIC_DIR = Path('dbs/us_foot_traffic/cities')

# US city to state FIPS mapping (for tract shapefiles)
CITY_STATE_FIPS = {
    'new_york': ['36', '34'],      # NY + NJ (metro area spans both)
    'washington_dc': ['11', '24', '51'],  # DC + MD + VA
    'atlanta': ['13'],             # GA
    'chicago': ['17', '18', '55'], # IL + IN + WI (metro area)
    'houston': ['48'],             # TX
    'phoenix': ['04'],             # AZ
}

print("Directories configured.")

/workspace
Directories configured.


## 1. Load Demographic Data with Geometries

In [3]:
# Load Swedish DeSO data WITH GEOMETRY for spatial join
deso_gdf = gpd.read_file(DESO_DIR / 'deso_harmonized_2024.gpkg')
print(f"Swedish DeSO zones: {len(deso_gdf)}")
print(f"CRS: {deso_gdf.crs}")

# Compute proportions for Sweden
# Birth: Sweden-born vs Other (outside EU), excluding Europe-born
deso_gdf['birth_native_foreign_total'] = deso_gdf['birth_sweden'] + deso_gdf['birth_other']
deso_gdf['p_native'] = deso_gdf['birth_sweden'] / deso_gdf['birth_native_foreign_total']
deso_gdf['p_foreign'] = deso_gdf['birth_other'] / deso_gdf['birth_native_foreign_total']
income_cols = ['income_q1_pct', 'income_q2_pct', 'income_q3_pct', 'income_q4_pct']

for col in income_cols:
    deso_gdf[col] = pd.to_numeric(deso_gdf[col], errors='coerce')
# Income: Normalize quartile percentages
deso_gdf['q_total'] = deso_gdf['income_q1_pct'] + deso_gdf['income_q2_pct'] + deso_gdf['income_q3_pct'] + deso_gdf['income_q4_pct']
deso_gdf['p_q1'] = deso_gdf['income_q1_pct'] / deso_gdf['q_total']
deso_gdf['p_q2'] = deso_gdf['income_q2_pct'] / deso_gdf['q_total']
deso_gdf['p_q3'] = deso_gdf['income_q3_pct'] / deso_gdf['q_total']
deso_gdf['p_q4'] = deso_gdf['income_q4_pct'] / deso_gdf['q_total']

# Also load as DataFrame for aggregation
deso_df = pd.DataFrame(deso_gdf.drop(columns='geometry'))

print("\nSweden proportions computed.")

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/geoenv/share/proj failed


Swedish DeSO zones: 6160
CRS: EPSG:3006

Sweden proportions computed.


In [4]:
# Load US Census data
us_census_df = pd.read_parquet(US_CENSUS_DIR / 'acs2024_tracts_study_cities.parquet')
print(f"US Census tracts: {len(us_census_df)}")
print(f"Cities: {us_census_df['city'].unique().tolist()}")

# Compute proportions for US
# Birth: Native vs Foreign
us_census_df['birth_total'] = us_census_df['native_born'] + us_census_df['foreign_born']
us_census_df['p_native'] = us_census_df['native_born'] / us_census_df['birth_total']
us_census_df['p_foreign'] = us_census_df['foreign_born'] / us_census_df['birth_total']

print(f"\nIncome quintile distribution:")
print(us_census_df['income_quintile'].value_counts())

US Census tracts: 12921
Cities: ['phoenix', 'washington_dc', 'atlanta', 'chicago', 'new_york', 'houston']

Income quintile distribution:
income_quintile
Q1_lowest     2586
Q5_highest    2585
Q2            2584
Q4            2584
Q3            2582
Name: count, dtype: int64


## 2. Spatial Join Functions for Residential Context

In [5]:
def assign_pois_to_deso(pois_df: pd.DataFrame, deso_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Spatial join POIs to DeSO zones.
    
    Parameters
    ----------
    pois_df : pd.DataFrame
        POIs with 'id', 'lat', 'lon' columns
    deso_gdf : gpd.GeoDataFrame
        DeSO zones with geometry (EPSG:3006)
    
    Returns
    -------
    pd.DataFrame
        POIs with added 'residential_deso' column
    """
    # Create GeoDataFrame from POIs (WGS84 -> SWEREF99 TM)
    pois_gdf = gpd.GeoDataFrame(
        pois_df,
        geometry=gpd.points_from_xy(pois_df['lon'], pois_df['lat']),
        crs='EPSG:4326'
    ).to_crs('EPSG:3006')
    
    # Spatial join
    joined = gpd.sjoin(
        pois_gdf,
        deso_gdf[['deso_code', 'geometry']],
        how='left',
        predicate='within'
    )
    
    # Handle POIs that didn't fall within any DeSO (use nearest)
    missing = joined['deso_code'].isna()
    if missing.any():
        print(f"  {missing.sum()} POIs outside DeSO boundaries, using nearest...")
        # For missing, find nearest DeSO centroid
        deso_centroids = deso_gdf.copy()
        deso_centroids['geometry'] = deso_centroids.geometry.centroid
        
        missing_pois = pois_gdf[missing]
        nearest = gpd.sjoin_nearest(
            missing_pois,
            deso_centroids[['deso_code', 'geometry']],
            how='left'
        )
        joined.loc[missing, 'deso_code'] = nearest['deso_code'].values
    
    result = pois_df.copy()
    result['residential_deso'] = joined['deso_code'].values
    return result


def load_us_tract_geometries(city: str) -> gpd.GeoDataFrame:
    """
    Load US census tract geometries for a city from shapefile zips.
    
    Parameters
    ----------
    city : str
        City name (must be in CITY_STATE_FIPS)
    
    Returns
    -------
    gpd.GeoDataFrame
        Tract geometries with GEOID column
    """
    state_fips = CITY_STATE_FIPS.get(city, [])
    if not state_fips:
        print(f"  Warning: No state FIPS mapping for {city}")
        return None
    
    all_tracts = []
    for fips in state_fips:
        shapefile = US_CENSUS_GEO_DIR / f'tl_2024_{fips}_tract.zip'
        if shapefile.exists():
            gdf = gpd.read_file(f'zip://{shapefile}')
            all_tracts.append(gdf)
            print(f"  Loaded {len(gdf)} tracts from state {fips}")
        else:
            print(f"  Warning: Shapefile not found for state {fips}")
    
    if not all_tracts:
        return None
    
    tracts_gdf = pd.concat(all_tracts, ignore_index=True)
    tracts_gdf = tracts_gdf.to_crs('EPSG:4326')
    return tracts_gdf


def assign_pois_to_tract_us(pois_df: pd.DataFrame, tracts_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Spatial join POIs to US census tracts.
    
    Parameters
    ----------
    pois_df : pd.DataFrame
        POIs with 'id', 'lat', 'lon' columns
    tracts_gdf : gpd.GeoDataFrame
        Census tracts with geometry and GEOID
    
    Returns
    -------
    pd.DataFrame
        POIs with added 'residential_tract' column
    """
    # Create GeoDataFrame from POIs
    pois_gdf = gpd.GeoDataFrame(
        pois_df,
        geometry=gpd.points_from_xy(pois_df['lon'], pois_df['lat']),
        crs='EPSG:4326'
    )
    
    # Ensure tracts are in same CRS
    if tracts_gdf.crs != 'EPSG:4326':
        tracts_gdf = tracts_gdf.to_crs('EPSG:4326')
    
    # Spatial join
    joined = gpd.sjoin(
        pois_gdf,
        tracts_gdf[['GEOID', 'geometry']],
        how='left',
        predicate='within'
    )
    
    # Handle POIs that didn't fall within any tract (use nearest)
    missing = joined['GEOID'].isna()
    if missing.any():
        print(f"  {missing.sum()} POIs outside tract boundaries, using nearest...")
        tract_centroids = tracts_gdf.copy()
        tract_centroids['geometry'] = tract_centroids.geometry.centroid
        
        missing_pois = pois_gdf[missing]
        nearest = gpd.sjoin_nearest(
            missing_pois,
            tract_centroids[['GEOID', 'geometry']],
            how='left'
        )
        joined.loc[missing, 'GEOID'] = nearest['GEOID'].values
    
    result = pois_df.copy()
    result['residential_tract'] = joined['GEOID'].values
    return result


print("Spatial join functions defined.")

Spatial join functions defined.


## 3. Aggregation Functions

In [6]:
def get_deso_composition(deso_code: str) -> dict:
    """
    Get composition for a single DeSO zone.
    """
    row = deso_df[deso_df['deso_code'] == deso_code]
    if len(row) == 0:
        return None
    row = row.iloc[0]
    return {
        'p_native': row['p_native'],
        'p_foreign': row['p_foreign'],
        'p_q1': row['p_q1'],
        'p_q2': row['p_q2'],
        'p_q3': row['p_q3'],
        'p_q4': row['p_q4'],
    }


def aggregate_sweden_composition(deso_codes: list, weights: list = None) -> dict:
    """
    Aggregate Swedish DeSO compositions with optional weights.
    """
    subset = deso_df[deso_df['deso_code'].isin(deso_codes)].copy()
    if len(subset) == 0:
        return None
    
    if weights is not None:
        weight_map = dict(zip(deso_codes, weights))
        subset['weight'] = subset['deso_code'].map(weight_map).fillna(0)
    else:
        subset['weight'] = subset['pop_total']
    
    total_weight = subset['weight'].sum()
    if total_weight == 0:
        return None
    
    # Weighted aggregation for birth
    native_sum = (subset['birth_sweden'] * subset['weight'] / subset['pop_total'].replace(0, np.nan)).sum()
    foreign_sum = (subset['birth_other'] * subset['weight'] / subset['pop_total'].replace(0, np.nan)).sum()
    birth_total = native_sum + foreign_sum
    
    if birth_total > 0:
        p_native = native_sum / birth_total
        p_foreign = foreign_sum / birth_total
    else:
        p_native = p_foreign = np.nan
    
    # Weighted aggregation for income
    income_weight = subset['income_n_persons'] * subset['weight'] / subset['pop_total'].replace(0, np.nan)
    q1_sum = (subset['income_q1_pct'] * income_weight / 100).sum()
    q2_sum = (subset['income_q2_pct'] * income_weight / 100).sum()
    q3_sum = (subset['income_q3_pct'] * income_weight / 100).sum()
    q4_sum = (subset['income_q4_pct'] * income_weight / 100).sum()
    q_total = q1_sum + q2_sum + q3_sum + q4_sum
    
    if q_total > 0:
        p_q1 = q1_sum / q_total
        p_q2 = q2_sum / q_total
        p_q3 = q3_sum / q_total
        p_q4 = q4_sum / q_total
    else:
        p_q1 = p_q2 = p_q3 = p_q4 = np.nan
    
    return {
        'p_native': p_native,
        'p_foreign': p_foreign,
        'p_q1': p_q1,
        'p_q2': p_q2,
        'p_q3': p_q3,
        'p_q4': p_q4
    }


def get_us_tract_composition(geoid: str) -> dict:
    """
    Get composition for a single US tract.
    Note: For single tract, income quartile is based on tract's quintile.
    """
    row = us_census_df[us_census_df['GEOID'] == geoid]
    if len(row) == 0:
        return None
    row = row.iloc[0]
    
    # Birth
    p_native = row['p_native']
    p_foreign = row['p_foreign']
    
    # Income: Single tract - assign to quartile based on quintile
    quintile = row['income_quintile']
    quintile_to_quartile = {
        'Q1_lowest': (1, 0, 0, 0),
        'Q2': (0, 1, 0, 0),
        'Q3': (0, 0, 1, 0),
        'Q4': (0, 0, 0, 1),
        'Q5_highest': (0, 0, 0, 1)
    }
    q = quintile_to_quartile.get(quintile, (0.25, 0.25, 0.25, 0.25))
    
    return {
        'p_native': p_native,
        'p_foreign': p_foreign,
        'p_q1': q[0],
        'p_q2': q[1],
        'p_q3': q[2],
        'p_q4': q[3]
    }


def aggregate_us_composition(geoids: list, weights: list = None) -> dict:
    """
    Aggregate US tract compositions with optional weights.
    """
    subset = us_census_df[us_census_df['GEOID'].isin(geoids)].copy()
    if len(subset) == 0:
        return None
    
    if weights is not None:
        weight_map = dict(zip(geoids, weights))
        subset['weight'] = subset['GEOID'].map(weight_map).fillna(0)
    else:
        subset['weight'] = subset['total_population']
    
    total_weight = subset['weight'].sum()
    if total_weight == 0:
        return None
    
    # Birth: weighted sum
    native_sum = (subset['native_born'] * subset['weight'] / subset['total_population'].replace(0, np.nan)).sum()
    foreign_sum = (subset['foreign_born'] * subset['weight'] / subset['total_population'].replace(0, np.nan)).sum()
    birth_total = native_sum + foreign_sum
    
    if birth_total > 0:
        p_native = native_sum / birth_total
        p_foreign = foreign_sum / birth_total
    else:
        p_native = p_foreign = np.nan
    
    # Income: map quintiles to quartiles
    quintile_to_quartile = {
        'Q1_lowest': 'q1',
        'Q2': 'q2',
        'Q3': 'q3',
        'Q4': 'q4',
        'Q5_highest': 'q4'
    }
    subset['quartile'] = subset['income_quintile'].map(quintile_to_quartile)
    
    quartile_weights = subset.groupby('quartile')['weight'].sum()
    q_total = quartile_weights.sum()
    
    if q_total > 0:
        p_q1 = quartile_weights.get('q1', 0) / q_total
        p_q2 = quartile_weights.get('q2', 0) / q_total
        p_q3 = quartile_weights.get('q3', 0) / q_total
        p_q4 = quartile_weights.get('q4', 0) / q_total
    else:
        p_q1 = p_q2 = p_q3 = p_q4 = np.nan
    
    return {
        'p_native': p_native,
        'p_foreign': p_foreign,
        'p_q1': p_q1,
        'p_q2': p_q2,
        'p_q3': p_q3,
        'p_q4': p_q4
    }


print("Aggregation functions defined.")

Aggregation functions defined.


## 4. Load Swedish Foot Traffic Data

In [7]:
# Load Swedish foot traffic data
se_traffic = pd.read_parquet(SE_TRAFFIC_DIR / 'sweden_weekly_patterns_2024.parquet')
print(f"Swedish foot traffic records: {len(se_traffic):,}")

# Check city distribution
print(f"\nTop 10 cities:")
print(se_traffic['CITY'].value_counts().head(10))

Swedish foot traffic records: 11,210,022

Top 10 cities:
CITY
Stockholm      957059
Göteborg       474269
Malmö          374674
Uppsala        218695
Linköping      162137
Örebro         139364
Västerås       137144
Helsingborg    134779
Norrköping     131481
Lund           124999
Name: count, dtype: int64


In [8]:
def parse_visitor_home_cbgs(cbgs_str: str) -> dict:
    """
    Parse VISITOR_HOME_CBGS_WEIGHTED string to dict.
    """
    if pd.isna(cbgs_str) or cbgs_str == '' or cbgs_str == '{}':
        return {}
    try:
        return json.loads(cbgs_str.replace("'", '"'))
    except:
        return {}

# Aggregate visitor homes per POI (across all weeks)
print("Aggregating visitor home locations per POI...")
se_poi_visitor_homes = {}

for _, row in tqdm(se_traffic.iterrows(), total=len(se_traffic), desc="Parsing visitor homes"):
    poi_id = row['PLACEKEY']
    homes = parse_visitor_home_cbgs(row['VISITOR_HOME_CBGS_WEIGHTED'])
    if poi_id not in se_poi_visitor_homes:
        se_poi_visitor_homes[poi_id] = {}
    for deso, weight in homes.items():
        se_poi_visitor_homes[poi_id][deso] = se_poi_visitor_homes[poi_id].get(deso, 0) + weight

print(f"\nPOIs with visitor home data: {len(se_poi_visitor_homes):,}")

Aggregating visitor home locations per POI...


Parsing visitor homes: 100%|██████████| 11210022/11210022 [07:17<00:00, 25616.37it/s]


POIs with visitor home data: 514,764


In [9]:
# Get unique POIs with their city
se_pois = se_traffic[['PLACEKEY', 'LATITUDE', 'LONGITUDE', 'CITY', 'REGION']].drop_duplicates(subset='PLACEKEY')
se_pois = se_pois.rename(columns={'PLACEKEY': 'id', 'LATITUDE': 'lat', 'LONGITUDE': 'lon', 'CITY': 'city', 'REGION': 'region'})
print(f"Unique Swedish POIs: {len(se_pois):,}")
print(f"Unique cities: {se_pois['city'].nunique():,}")

Unique Swedish POIs: 514,764
Unique cities: 4,319


## 5. Process Sweden Data

For each county with transit catchment data, compute diversity metrics.

In [10]:
# Swedish county codes and region mapping
SWEDEN_COUNTIES = ['01', '03', '04', '05', '06', '07', '08', '09', '10', '12', '13', '14', '17', '18', '19']

COUNTY_TO_REGION = {
    '01': ['Stockholm County'],
    '03': ['Uppsala County'],
    '04': ['Södermanland County', 'S\xf6dermanland County'],
    '05': ['Östergötland County', '\xd6sterg\xf6tland County'],
    '06': ['Jönköping County', 'J\xf6nk\xf6ping County'],
    '07': ['Kronoberg County'],
    '08': ['Kalmar County'],
    '09': ['Gotland County'],
    '10': ['Blekinge County'],
    '12': ['Skåne County', 'Sk\xe5ne County'],
    '13': ['Halland County'],
    '14': ['Västra Götaland County', 'V\xe4stra G\xf6taland County'],
    '17': ['Värmland County', 'V\xe4rmland County'],
    '18': ['Örebro County', '\xd6rebro County'],
    '19': ['Västmanland County', 'V\xe4stmanland County'],
}

# Check which counties have transit catchment data
available_se_catchment = []
for c in SWEDEN_COUNTIES:
    catchment_file = ROUTING_DIR / f'sweden_c{c}' / 'catchment_summary_walk_transit_45min.parquet'
    if catchment_file.exists():
        available_se_catchment.append(c)
        print(f"County {c}: Transit catchment available")
    else:
        print(f"County {c}: No transit catchment data")

print(f"\nCounties with catchment data: {len(available_se_catchment)}")

County 01: Transit catchment available
County 03: Transit catchment available
County 04: Transit catchment available
County 05: Transit catchment available
County 06: Transit catchment available
County 07: Transit catchment available
County 08: Transit catchment available
County 09: Transit catchment available
County 10: No transit catchment data
County 12: Transit catchment available
County 13: Transit catchment available
County 14: Transit catchment available
County 17: Transit catchment available
County 18: Transit catchment available
County 19: Transit catchment available

Counties with catchment data: 14


In [11]:
def process_sweden_county(county_code: str, city_filter: list = None) -> pd.DataFrame:
    """
    Process a single Swedish county: compute diversity metrics for all POIs.
    
    Parameters
    ----------
    county_code : str
        County code (e.g., '01' for Stockholm)
    city_filter : list, optional
        List of city names to include. If None, include all.
    
    Returns
    -------
    pd.DataFrame with diversity metrics
    """
    print(f"\nProcessing county {county_code}...")
    
    # Get POIs for this county
    region_names = COUNTY_TO_REGION[county_code]
    county_pois = se_pois[se_pois['region'].isin(region_names)].copy()
    
    # Apply city filter if specified
    if city_filter:
        county_pois = county_pois[county_pois['city'].isin(city_filter)]
        print(f"  POIs after city filter: {len(county_pois):,}")
    else:
        print(f"  POIs: {len(county_pois):,}")
    
    if len(county_pois) == 0:
        return None
    
    # Load transit catchment results
    catchment_file = ROUTING_DIR / f'sweden_c{county_code}' / 'catchment_summary_walk_transit_45min.parquet'
    if not catchment_file.exists():
        print(f"  No catchment data, skipping...")
        return None
    
    catchment = pd.read_parquet(catchment_file)
    print(f"  Catchment results: {len(catchment):,}")
    
    # Spatial join: assign POIs to residential DeSO
    print(f"  Performing spatial join for residential context...")
    county_pois = assign_pois_to_deso(county_pois, deso_gdf)
    print(f"  POIs with residential DeSO: {county_pois['residential_deso'].notna().sum():,}")
    
    # Create catchment lookup
    catchment_dict = catchment.set_index('poi_id')['reachable_tract_ids'].to_dict()
    
    # Process each POI
    results = []
    
    for _, poi in tqdm(county_pois.iterrows(), total=len(county_pois), desc=f"County {county_code}"):
        poi_id = poi['id']
        
        result = {
            'poi_id': poi_id,
            'lat': poi['lat'],
            'lon': poi['lon'],
            'city': poi['city'],
            'region': poi['region'],
            'county_code': county_code
        }
        
        # 1. RESIDENTIAL: Use spatially joined DeSO
        res_deso = poi['residential_deso']
        if pd.notna(res_deso):
            residential_comp = get_deso_composition(res_deso)
            result['residential_deso'] = res_deso
        else:
            residential_comp = None
            result['residential_deso'] = None
        
        if residential_comp:
            res_metrics = compute_diversity_metrics(residential_comp)
            for k, v in res_metrics.items():
                result[f'residential_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'residential_{k}'] = np.nan
        
        # 2. CATCHMENT: Aggregate all reachable DeSO zones
        reachable_str = catchment_dict.get(poi_id, '')
        if reachable_str and pd.notna(reachable_str):
            reachable = [r for r in reachable_str.split('|') if r]
            catchment_comp = aggregate_sweden_composition(reachable) if reachable else None
            n_tracts = len(reachable)
        else:
            catchment_comp = None
            n_tracts = 0
        
        if catchment_comp:
            catch_metrics = compute_diversity_metrics(catchment_comp)
            for k, v in catch_metrics.items():
                result[f'catchment_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'catchment_{k}'] = np.nan
        result['catchment_n_tracts'] = n_tracts
        
        # 3. VISITORS: Aggregate based on visitor home DeSO
        if poi_id in se_poi_visitor_homes and se_poi_visitor_homes[poi_id]:
            homes = se_poi_visitor_homes[poi_id]
            deso_codes = list(homes.keys())
            weights = list(homes.values())
            visitor_comp = aggregate_sweden_composition(deso_codes, weights)
            n_home_tracts = len(homes)
        else:
            visitor_comp = None
            n_home_tracts = 0
        
        if visitor_comp:
            vis_metrics = compute_diversity_metrics(visitor_comp)
            for k, v in vis_metrics.items():
                result[f'visitor_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'visitor_{k}'] = np.nan
        result['visitor_n_home_tracts'] = n_home_tracts
        
        results.append(result)
    
    return pd.DataFrame(results)

In [12]:
# Define major cities to focus on (optional filter)
# Set to None to include all POIs
SWEDEN_MAJOR_CITIES = [
    'Stockholm', 'Göteborg', 'Malmö', 'Uppsala', 'Linköping', 'Örebro',
    'Västerås', 'Helsingborg', 'Norrköping', 'Lund', 'Jönköping', 'Karlstad',
    'Borås', 'Halmstad', 'Södertälje', 'Eskilstuna', 'Växjö', 'Trollhättan',
    # Add encoding variants
    'G\xf6teborg', 'Malm\xf6', '\xd6rebro', 'V\xe4ster\xe5s', 'Norrk\xf6ping',
    'J\xf6nk\xf6ping', 'S\xf6dert\xe4lje', 'V\xe4xj\xf6', 'Trollh\xe4ttan'
]

# Set to None to process all POIs (no city filter)
CITY_FILTER = SWEDEN_MAJOR_CITIES  # or SWEDEN_MAJOR_CITIES to filter

print(f"City filter: {'All cities' if CITY_FILTER is None else f'{len(set(CITY_FILTER))} cities'}")

City filter: 18 cities


In [13]:
# Process all counties with catchment data
sweden_results = []

for county_code in available_se_catchment:
    result = process_sweden_county(county_code, city_filter=CITY_FILTER)
    if result is not None and len(result) > 0:
        sweden_results.append(result)

if sweden_results:
    sweden_diversity = pd.concat(sweden_results, ignore_index=True)
    print(f"\nTotal Sweden POIs processed: {len(sweden_diversity):,}")
    print(f"Cities represented: {sweden_diversity['city'].nunique()}")
    
    # Save
    output_path = ROUTING_DIR / 'sweden_poi_diversity_metrics.parquet'
    sweden_diversity.to_parquet(output_path)
    print(f"Saved: {output_path}")
else:
    print("No Sweden results to save.")
    sweden_diversity = None


Processing county 01...
  POIs after city filter: 38,910
  Catchment results: 124,105
  Performing spatial join for residential context...
  POIs with residential DeSO: 38,910


County 01: 100%|██████████| 38910/38910 [04:10<00:00, 155.26it/s]



Processing county 03...
  POIs after city filter: 8,175
  Catchment results: 18,124
  Performing spatial join for residential context...
  POIs with residential DeSO: 8,175


County 03: 100%|██████████| 8175/8175 [00:48<00:00, 168.02it/s]



Processing county 04...
  POIs after city filter: 3,419
  Catchment results: 14,093
  Performing spatial join for residential context...
  POIs with residential DeSO: 3,419


...


Processing county 18...
  POIs after city filter: 5,651
  Catchment results: 13,469
  Performing spatial join for residential context...
  POIs with residential DeSO: 5,651


County 18: 100%|██████████| 5651/5651 [00:33<00:00, 170.58it/s]



Processing county 19...
  POIs after city filter: 5,662
  Catchment results: 11,919
  Performing spatial join for residential context...
  POIs with residential DeSO: 5,662


County 19: 100%|██████████| 5662/5662 [00:32<00:00, 172.45it/s]



Total Sweden POIs processed: 132,469
Cities represented: 18
Saved: dbs/routing/sweden_poi_diversity_metrics.parquet


## 6. Process US Data

In [14]:
# US cities
US_CITIES = ['new_york', 'washington_dc', 'atlanta', 'chicago', 'houston', 'phoenix']

# Check which cities have transit catchment data
available_us_catchment = []
for city in US_CITIES:
    catchment_file = ROUTING_DIR / city / 'catchment_summary_walk_transit_45min.parquet'
    if catchment_file.exists():
        available_us_catchment.append(city)
        print(f"{city}: Transit catchment available")
    else:
        print(f"{city}: No transit catchment data")

print(f"\nCities with catchment data: {len(available_us_catchment)}")

new_york: Transit catchment available
washington_dc: Transit catchment available
atlanta: Transit catchment available
chicago: No transit catchment data
houston: No transit catchment data
phoenix: No transit catchment data

Cities with catchment data: 3


In [15]:
def load_us_city_traffic(city: str) -> pd.DataFrame:
    """
    Load and concatenate US city foot traffic data.
    """
    city_dir = US_TRAFFIC_DIR / city
    if not city_dir.exists():
        return None
    
    files = list(city_dir.glob('*.parquet'))
    if not files:
        return None
    
    dfs = [pd.read_parquet(f) for f in files]
    return pd.concat(dfs, ignore_index=True)


def process_us_city(city: str) -> pd.DataFrame:
    """
    Process a single US city: compute diversity metrics for all POIs.
    
    US foot traffic schema:
    - ID_STORE: POI identifier (needs string conversion for matching)
    - VISITOR_HOME_CBGS: JSON dict of CBG -> visit count
    """
    print(f"\nProcessing {city}...")
    
    # Load POI locations
    poi_file = ROUTING_DIR / f'{city}_poi_locations.csv'
    if not poi_file.exists():
        print(f"  No POI file found")
        return None
    pois = pd.read_csv(poi_file)
    # CRITICAL: Convert id to string for consistent matching
    pois['id'] = pois['id'].astype(str)
    print(f"  POIs: {len(pois):,}")
    
    # Load transit catchment
    catchment_file = ROUTING_DIR / city / 'catchment_summary_walk_transit_45min.parquet'
    if not catchment_file.exists():
        print(f"  No catchment data")
        return None
    catchment = pd.read_parquet(catchment_file)
    # CRITICAL: Ensure poi_id is string for consistent matching
    catchment['poi_id'] = catchment['poi_id'].astype(str)
    print(f"  Catchment results: {len(catchment):,}")
    
    # Load tract geometries for spatial join
    print(f"  Loading tract geometries...")
    tracts_gdf = load_us_tract_geometries(city)
    if tracts_gdf is None:
        print(f"  No tract geometries, cannot perform spatial join")
        return None
    print(f"  Total tracts loaded: {len(tracts_gdf):,}")
    
    # Spatial join: assign POIs to residential tract
    print(f"  Performing spatial join for residential context...")
    pois = assign_pois_to_tract_us(pois, tracts_gdf)
    print(f"  POIs with residential tract: {pois['residential_tract'].notna().sum():,}")
    
    # Load foot traffic
    traffic = load_us_city_traffic(city)
    if traffic is None:
        print(f"  No traffic data")
        return None
    print(f"  Traffic records: {len(traffic):,}")
    
    # Aggregate visitor homes per POI
    # US uses ID_STORE as POI identifier, VISITOR_HOME_CBGS for visitor homes
    print(f"  Aggregating visitor home locations...")
    poi_visitor_homes = {}
    
    for _, row in traffic.iterrows():
        # CRITICAL: Convert to string for consistent matching
        poi_id = str(row['ID_STORE'])
        if 'VISITOR_HOME_CBGS' in row and pd.notna(row['VISITOR_HOME_CBGS']):
            try:
                homes = json.loads(str(row['VISITOR_HOME_CBGS']).replace("'", '"'))
                # Convert CBG to tract (first 11 chars)
                tract_homes = {}
                for cbg, count in homes.items():
                    tract = str(cbg)[:11]
                    tract_homes[tract] = tract_homes.get(tract, 0) + count
                
                if poi_id not in poi_visitor_homes:
                    poi_visitor_homes[poi_id] = {}
                for tract, count in tract_homes.items():
                    poi_visitor_homes[poi_id][tract] = poi_visitor_homes[poi_id].get(tract, 0) + count
            except:
                pass
    
    print(f"  POIs with visitor data: {len(poi_visitor_homes):,}")
    
    # Create catchment lookup (keys are now strings)
    catchment_dict = catchment.set_index('poi_id')['reachable_tract_ids'].to_dict()
    
    # Debug: Check matching
    poi_ids_set = set(pois['id'])
    catchment_overlap = len(poi_ids_set & set(catchment_dict.keys()))
    visitor_overlap = len(poi_ids_set & set(poi_visitor_homes.keys()))
    print(f"  POIs with catchment match: {catchment_overlap:,}")
    print(f"  POIs with visitor match: {visitor_overlap:,}")
    
    # Process each POI
    results = []
    
    for _, poi in tqdm(pois.iterrows(), total=len(pois), desc=city):
        poi_id = poi['id']  # Already string from astype(str) above
        
        result = {
            'poi_id': poi_id,
            'lat': poi['lat'],
            'lon': poi['lon'],
            'city': city
        }
        
        # 1. RESIDENTIAL: Use spatially joined tract
        res_tract = poi['residential_tract']
        if pd.notna(res_tract):
            residential_comp = get_us_tract_composition(res_tract)
            result['residential_tract'] = res_tract
        else:
            residential_comp = None
            result['residential_tract'] = None
        
        if residential_comp:
            res_metrics = compute_diversity_metrics(residential_comp)
            for k, v in res_metrics.items():
                result[f'residential_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'residential_{k}'] = np.nan
        
        # 2. CATCHMENT (poi_id is string, catchment_dict keys are strings)
        reachable_str = catchment_dict.get(poi_id, '')
        if reachable_str and pd.notna(reachable_str):
            reachable = [r for r in reachable_str.split('|') if r]
            catchment_comp = aggregate_us_composition(reachable) if reachable else None
            n_tracts = len(reachable)
        else:
            catchment_comp = None
            n_tracts = 0
            reachable = []
        
        if catchment_comp:
            catch_metrics = compute_diversity_metrics(catchment_comp)
            for k, v in catch_metrics.items():
                result[f'catchment_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'catchment_{k}'] = np.nan
        result['catchment_n_tracts'] = n_tracts
        
        # 3. VISITORS (poi_id is string, poi_visitor_homes keys are strings)
        if poi_id in poi_visitor_homes and poi_visitor_homes[poi_id]:
            homes = poi_visitor_homes[poi_id]
            tracts = list(homes.keys())
            weights = list(homes.values())
            visitor_comp = aggregate_us_composition(tracts, weights)
            n_home_tracts = len(homes)
        else:
            visitor_comp = None
            n_home_tracts = 0
        
        if visitor_comp:
            vis_metrics = compute_diversity_metrics(visitor_comp)
            for k, v in vis_metrics.items():
                result[f'visitor_{k}'] = v
        else:
            for k in ['entropy_birth', 'entropy_birth_norm', 'entropy_income', 'entropy_income_norm', 'ice_birth', 'ice_income']:
                result[f'visitor_{k}'] = np.nan
        result['visitor_n_home_tracts'] = n_home_tracts
        
        results.append(result)
    
    return pd.DataFrame(results)

In [16]:
# Process all US cities with catchment data
us_results = []

for city in available_us_catchment:
    result = process_us_city(city)
    if result is not None and len(result) > 0:
        us_results.append(result)

if us_results:
    us_diversity = pd.concat(us_results, ignore_index=True)
    print(f"\nTotal US POIs processed: {len(us_diversity):,}")
    
    # Save
    output_path = ROUTING_DIR / 'us_poi_diversity_metrics.parquet'
    us_diversity.to_parquet(output_path)
    print(f"Saved: {output_path}")
else:
    print("No US results to save.")
    us_diversity = None


Processing new_york...
  POIs: 290,270
  Catchment results: 263,840
  Loading tract geometries...
  Loaded 5411 tracts from state 36
  Loaded 2181 tracts from state 34
  Total tracts loaded: 7,592
  Performing spatial join for residential context...
  952 POIs outside tract boundaries, using nearest...


/tmp/ipykernel_19151/3732000522.py:130: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tract_centroids['geometry'] = tract_centroids.geometry.centroid
/opt/conda/envs/geoenv/lib/python3.11/site-packages/geopandas/array.py:407: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


  POIs with residential tract: 290,270
  Traffic records: 14,139,105
  Aggregating visitor home locations...
  POIs with visitor data: 273,055
  POIs with catchment match: 263,840
  POIs with visitor match: 273,055


new_york: 100%|██████████| 290270/290270 [43:02<00:00, 112.38it/s]



Processing washington_dc...
  POIs: 84,650
  Catchment results: 84,179
  Loading tract geometries...
  Loaded 206 tracts from state 11
  Loaded 1475 tracts from state 24
  Loaded 2198 tracts from state 51
  Total tracts loaded: 3,879
  Performing spatial join for residential context...
  901 POIs outside tract boundaries, using nearest...
  POIs with residential tract: 84,650


...


Processing atlanta...
  POIs: 102,181
  Catchment results: 102,181
  Loading tract geometries...
  Loaded 2796 tracts from state 13
  Total tracts loaded: 2,796
  Performing spatial join for residential context...
  1 POIs outside tract boundaries, using nearest...
  POIs with residential tract: 102,181


/tmp/ipykernel_19151/3732000522.py:130: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tract_centroids['geometry'] = tract_centroids.geometry.centroid
/opt/conda/envs/geoenv/lib/python3.11/site-packages/geopandas/array.py:407: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


  Traffic records: 4,781,605
  Aggregating visitor home locations...
  POIs with visitor data: 92,947
  POIs with catchment match: 102,181
  POIs with visitor match: 92,947


atlanta: 100%|██████████| 102181/102181 [13:22<00:00, 127.39it/s]



Total US POIs processed: 477,101
Saved: dbs/routing/us_poi_diversity_metrics.parquet


## 7. Summary Statistics

In [17]:
def summarize_diversity_metrics(df: pd.DataFrame, name: str):
    """
    Print summary statistics for diversity metrics.
    """
    print(f"\n{'='*70}")
    print(f"{name} DIVERSITY METRICS SUMMARY")
    print(f"{'='*70}")
    print(f"Total POIs: {len(df):,}")
    
    metrics = ['entropy_birth_norm', 'entropy_income_norm', 'ice_birth', 'ice_income']
    contexts = ['residential', 'catchment', 'visitor']
    
    for metric in metrics:
        print(f"\n{metric}:")
        for ctx in contexts:
            col = f'{ctx}_{metric}'
            if col in df.columns:
                valid = df[col].dropna()
                if len(valid) > 0:
                    print(f"  {ctx:12}: mean={valid.mean():.3f}, std={valid.std():.3f}, "
                          f"min={valid.min():.3f}, max={valid.max():.3f}, n={len(valid):,}")
                else:
                    print(f"  {ctx:12}: no valid data")

# Summarize
if sweden_diversity is not None:
    summarize_diversity_metrics(sweden_diversity, "SWEDEN")

if us_diversity is not None:
    summarize_diversity_metrics(us_diversity, "US")


SWEDEN DIVERSITY METRICS SUMMARY
Total POIs: 132,469

entropy_birth_norm:
  residential : mean=0.536, std=0.216, min=0.074, max=1.000, n=132,469
  catchment   : mean=0.565, std=0.196, min=0.091, max=1.000, n=116,045
  visitor     : mean=0.589, std=0.161, min=0.055, max=1.000, n=132,469

entropy_income_norm:
  residential : mean=0.946, std=0.063, min=0.203, max=1.000, n=132,469
  catchment   : mean=0.951, std=0.052, min=0.203, max=1.000, n=116,045
  visitor     : mean=0.982, std=0.029, min=0.253, max=1.000, n=132,469

ice_birth:
  residential : mean=0.700, std=0.248, min=-0.931, max=0.982, n=132,469
  catchment   : mean=0.687, std=0.231, min=-0.931, max=0.977, n=116,045
  visitor     : mean=0.693, std=0.154, min=-0.504, max=0.987, n=132,469

ice_income:
  residential : mean=0.053, std=0.363, min=-1.000, max=0.811, n=132,469
  catchment   : mean=0.025, std=0.348, min=-1.000, max=0.811, n=116,045
  visitor     : mean=0.020, std=0.230, min=-0.978, max=0.855, n=132,469

US DIVERSITY METRIC

In [18]:
def compare_catchment_visitor(df: pd.DataFrame, name: str):
    """
    Compare catchment potential vs actual visitor diversity.
    """
    print(f"\n{'='*70}")
    print(f"{name}: CATCHMENT vs VISITOR COMPARISON")
    print(f"{'='*70}")
    
    for metric in ['entropy_birth_norm', 'entropy_income_norm', 'ice_birth', 'ice_income']:
        catch_col = f'catchment_{metric}'
        vis_col = f'visitor_{metric}'
        
        if catch_col in df.columns and vis_col in df.columns:
            valid = df[[catch_col, vis_col]].dropna()
            if len(valid) > 0:
                diff = valid[vis_col] - valid[catch_col]
                corr = valid[catch_col].corr(valid[vis_col])
                print(f"\n{metric}:")
                print(f"  N valid pairs: {len(valid):,}")
                print(f"  Correlation: {corr:.3f}")
                print(f"  Mean diff (visitor - catchment): {diff.mean():.3f}")
                print(f"  % visitor > catchment: {(diff > 0).mean()*100:.1f}%")
                print(f"  % visitor < catchment: {(diff < 0).mean()*100:.1f}%")

if sweden_diversity is not None:
    compare_catchment_visitor(sweden_diversity, "SWEDEN")

if us_diversity is not None:
    compare_catchment_visitor(us_diversity, "US")


SWEDEN: CATCHMENT vs VISITOR COMPARISON

entropy_birth_norm:
  N valid pairs: 116,045
  Correlation: 0.357
  Mean diff (visitor - catchment): 0.029
  % visitor > catchment: 58.7%
  % visitor < catchment: 41.3%

entropy_income_norm:
  N valid pairs: 116,045
  Correlation: 0.236
  Mean diff (visitor - catchment): 0.031
  % visitor > catchment: 78.9%
  % visitor < catchment: 21.1%

ice_birth:
  N valid pairs: 116,045
  Correlation: 0.376
  Mean diff (visitor - catchment): 0.003
  % visitor > catchment: 41.5%
  % visitor < catchment: 58.4%

ice_income:
  N valid pairs: 116,045
  Correlation: 0.502
  Mean diff (visitor - catchment): -0.001
  % visitor > catchment: 45.9%
  % visitor < catchment: 54.1%

US: CATCHMENT vs VISITOR COMPARISON

entropy_birth_norm:
  N valid pairs: 202,727
  Correlation: 0.820
  Mean diff (visitor - catchment): 0.009
  % visitor > catchment: 54.4%
  % visitor < catchment: 44.9%

entropy_income_norm:
  N valid pairs: 202,727
  Correlation: 0.281
  Mean diff (visito

In [19]:
# City-level summary for Sweden
if sweden_diversity is not None:
    print("\n" + "="*70)
    print("SWEDEN: CITY-LEVEL SUMMARY")
    print("="*70)
    
    city_summary = sweden_diversity.groupby('city').agg({
        'poi_id': 'count',
        'catchment_entropy_birth_norm': 'mean',
        'visitor_entropy_birth_norm': 'mean',
        'catchment_ice_birth': 'mean',
        'visitor_ice_birth': 'mean',
    }).round(3)
    city_summary.columns = ['n_pois', 'catch_H_birth', 'vis_H_birth', 'catch_ICE_birth', 'vis_ICE_birth']
    city_summary = city_summary.sort_values('n_pois', ascending=False).head(20)
    print(city_summary)


SWEDEN: CITY-LEVEL SUMMARY
             n_pois  catch_H_birth  vis_H_birth  catch_ICE_birth  \
city                                                               
Stockholm     35948          0.453        0.583            0.792   
Göteborg      16806          0.581        0.606            0.683   
Malmö         12680          0.701        0.652            0.565   
Uppsala        8170          0.586        0.590            0.682   
Västerås       5666          0.659        0.610            0.601   
Örebro         5657          0.554        0.506            0.701   
Linköping      5634          0.547        0.526            0.697   
Helsingborg    5488          0.665        0.597            0.595   
Lund           4900          0.574        0.575            0.710   
Norrköping     4652          0.630        0.562            0.649   
Karlstad       3902          0.451        0.462            0.777   
Halmstad       3816          0.520        0.546            0.714   
Jönköping      3728 

## 8. Output Summary

### Files Created

| File | Description |
|------|-------------|
| `dbs/routing/sweden_poi_diversity_metrics.parquet` | Swedish POI diversity metrics |
| `dbs/routing/us_poi_diversity_metrics.parquet` | US POI diversity metrics |

### Spatial Join Data Sources

| Country | Residential Context | Source |
|---------|---------------------|--------|
| Sweden | DeSO zones | `dbs/deso/deso_harmonized_2024.gpkg` |
| US | Census tracts | `dbs/census_geo/tl_2024_XX_tract.zip` |

### Columns

| Column | Description |
|--------|-------------|
| `poi_id` | POI identifier |
| `lat`, `lon` | Coordinates |
| `city` | City name |
| `county_code` / `region` | Geographic region (Sweden only) |
| `residential_deso` / `residential_tract` | Tract where POI is located (via spatial join) |
| `residential_*` | Metrics for POI's home tract |
| `catchment_*` | Metrics for transit-reachable area |
| `visitor_*` | Metrics for actual visitor home locations |
| `*_n_tracts` | Number of tracts in aggregation |

### Metrics

| Metric | Range | Interpretation |
|--------|-------|----------------|
| `entropy_birth` | 0 to log(2) | Binary entropy (native/foreign) |
| `entropy_birth_norm` | 0 to 1 | Normalized (1 = max diversity) |
| `entropy_income` | 0 to log(4) | 4-quartile entropy |
| `entropy_income_norm` | 0 to 1 | Normalized (1 = max diversity) |
| `ice_birth` | -1 to +1 | +1=all native, -1=all foreign |
| `ice_income` | -1 to +1 | +1=all Q4 (high), -1=all Q1 (low) |